In [4]:
import json, random, time, logging
from datetime import datetime, timezone
from confluent_kafka import Producer, KafkaException

logging.basicConfig(level=logging.INFO)
log = logging.getLogger("air_quality")

BOOTSTRAP = "localhost:9092,localhost:9094,localhost:9096"
TOPIC = "urbanpulse.air_quality"
SENSOR_IDS = [f"AQ-{i:04d}" for i in range(1, 601)]
ZONES = [f"ZONE-{z}" for z in "ABCDEFGH"]
NULL_AQI_RATE = 0.05
MAX_RETRIES = 5

producer = Producer({"bootstrap.servers": BOOTSTRAP, "acks": "all", "enable.idempotence": True})

def make_reading(sensor_id):
    failed = random.random() < NULL_AQI_RATE
    pm25 = round(random.uniform(10, 400), 1)
    aqi = None if failed else round(pm25 * random.uniform(0.9, 1.3))
    if failed:
        log.warning(f"Sensor {sensor_id} NULL AQI (simulated timeout) - forwarding for DLQ handling")
    return {"sensor_id": sensor_id, "zone": random.choice(ZONES), "pm25": pm25,
            "pm10": round(pm25 * 1.3, 1), "no2": round(random.uniform(5, 120), 1),
            "aqi": aqi, "timestamp": datetime.now(timezone.utc).isoformat()}

def produce_with_retry(event, key):
    attempt = 0
    while attempt < MAX_RETRIES:
        try:
            producer.produce(TOPIC, key=key.encode(), value=json.dumps(event).encode())
            producer.poll(0)
            return True
        except BufferError:
            producer.poll(1); attempt += 1
        except KafkaException as e:
            attempt += 1
            log.error(f"attempt {attempt} failed: {e}; retrying in {2**attempt}s")
            time.sleep(2 ** attempt)
    log.critical(f"giving up on {event['sensor_id']}")
    return False

def run(duration_sec=300, target_rate=20):
    end = time.time() + duration_sec
    sent = nulls = 0
    while time.time() < end:
        sid = random.choice(SENSOR_IDS)
        ev = make_reading(sid)
        nulls += ev["aqi"] is None
        produce_with_retry(ev, sid)
        sent += 1
        time.sleep(1.0 / target_rate)
    producer.flush(10)
    log.info(f"sent={sent}, null-AQI={nulls} ({nulls/sent:.1%})")

run()

INFO:air_quality:sent=5957, null-AQI=284 (4.8%)
